# Pipeline Final de Inferencia Thoracolumbar Explicado - Colab

Este notebook empaqueta la mejor version actual del pipeline para inferencia sobre
nuevas radiografias.

## Pipeline final actual

1. modelo binario localiza la columna
2. modelo multiclase segmenta `T1..T12 + L1..L5`
3. estimador especializado predice la ultima vertebra visible
4. clipping anatomico recorta la mascara final

## Objetivo del notebook

Permitir inferencia reproducible sobre una carpeta de imagenes nuevas y exportar:

- etiquetas presentes antes y despues del clipping
- ultima vertebra visible estimada
- visualizaciones de apoyo
- mascaras exportables

## Nota importante

Esta version asume que la primera vertebra visible del problema util suele comenzar
en `T1`, y concentra la correccion anatomica en la **ultima vertebra visible**,
porque ese fue el principal cuello de botella observado durante el proyecto.

## 0. Preparacion de Colab

In [ ]:
import os
from pathlib import Path


PROJECT_ROOT_CANDIDATES = [
    Path.cwd() / "data" / "Scoliosis_Dataset",
    Path.cwd().parent / "data" / "Scoliosis_Dataset",
    Path.cwd().parent.parent / "data" / "Scoliosis_Dataset",
    Path.cwd() / "Scoliosis_Dataset",
    Path.cwd().parent / "Scoliosis_Dataset",
]

PROJECT_ROOT = next((path for path in PROJECT_ROOT_CANDIDATES if path.exists()), None)
if PROJECT_ROOT is None:
    searched = "\n".join(str(path) for path in PROJECT_ROOT_CANDIDATES)
    raise FileNotFoundError(
        "No se encontro data/Scoliosis_Dataset. Rutas revisadas:\n" + searched
    )

os.chdir(PROJECT_ROOT)
print('Working directory:', Path.cwd())

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# CAMBIA ESTA RUTA SOLO SI TU CARPETA ESTA EN OTRO LUGAR
PROJECT_ROOT = Path("/content/drive/MyDrive/DataRadriografias")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"No existe la carpeta: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)

print("Working directory:", Path.cwd())

Mounted at /content/drive
Working directory: /content/drive/MyDrive/DataRadriografias


## 1. Configuracion de inferencia

Ajusta solo esta seccion para usar el pipeline sobre tus imagenes nuevas.

In [14]:
from __future__ import annotations

import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage as ndi

import torch
import torch.nn as nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

#CWD = Path.cwd()

ROOT_CANDIDATES = [
    Path.cwd() / "data" / "Scoliosis_Dataset",
    Path.cwd().parent / "data" / "Scoliosis_Dataset",
    Path.cwd().parent.parent / "data" / "Scoliosis_Dataset",
    Path.cwd() / "Scoliosis_Dataset",
    Path.cwd().parent / "Scoliosis_Dataset",
]

REQUIRED_FILES = [
    "indice_dataset.csv",
    "diccionario_etiquetas_T1_T12_L1_L5.json",
]


def is_valid_root(path: Path) -> bool:
    return path.exists() and all((path / rel).exists() for rel in REQUIRED_FILES)


ROOT = next((p for p in ROOT_CANDIDATES if is_valid_root(p)), None)
assert ROOT is not None, (
    "No se pudo localizar la carpeta data/Scoliosis_Dataset con los archivos esperados. "
    f"Directorio actual: {CWD}"
)

ROOT = ROOT.resolve()
PROJECT_DIR_CANDIDATES = [ROOT.parent.parent, *ROOT.parents, CWD, *CWD.parents]
PROJECT_DIR = next(
    (
        path
        for path in PROJECT_DIR_CANDIDATES
        if (path / "notebooks").exists() and (path / "data").exists()
    ),
    ROOT.parent.parent,
)
REPORTS_DIR = PROJECT_DIR / "reports"
ANALYSIS_DIR = REPORTS_DIR / "analysis_outputs"
OUTPUTS_DIR = PROJECT_DIR / "outputs"
MODELS_DIR = PROJECT_DIR / "models"
for directory in [REPORTS_DIR, ANALYSIS_DIR, OUTPUTS_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SEARCH_BASES = [CWD.resolve(), ROOT, *ROOT.parents]


def resolve_dataset_path(path_value: str | Path) -> Path:
    raw = Path(str(path_value))
    candidates: list[Path] = []

    if raw.is_absolute():
        candidates.append(raw)
    else:
        candidates.append(raw)
        candidates.extend(base / raw for base in SEARCH_BASES)

    parts = raw.parts
    if "Scoliosis_Dataset" in parts:
            idx = parts.index("Scoliosis_Dataset")
            trimmed = Path(*parts[idx + 1 :])
            if str(trimmed) not in {"", "."}:
                candidates.append(ROOT / trimmed)

    seen: set[str] = set()
    unique_candidates: list[Path] = []
    for candidate in candidates:
        key = str(candidate)
        if key not in seen:
            seen.add(key)
            unique_candidates.append(candidate)

    for candidate in unique_candidates:
        if candidate.exists():
            return candidate.resolve()

    return unique_candidates[-1].resolve()


INDEX_PATH = ROOT / "indice_dataset.csv"
DICT_PATH = ROOT / "diccionario_etiquetas_T1_T12_L1_L5.json"
MANIFEST_PATH = PROJECT_DIR / "reports" / "analysis_outputs" / "training_manifest_thoracolumbar.csv"
BINARY_GROUP_MAP_PATH = PROJECT_DIR / "reports" / "analysis_outputs" / "training_runs_binary_thoracolumbar" / "binary_spine_group_partition_map.csv"
BINARY_MODEL_PATH = PROJECT_DIR / "models" / "binary_spine_thoracolumbar_best.pt"
MULTICLASS_MODEL_PATH = PROJECT_DIR / "models" / "thoracolumbar_partial_cascade_explained_best.pt"
LAST_VISIBLE_MODEL_PATH = PROJECT_DIR / "models" / "last_visible_estimator_thoracolumbar_best.pt"
BINARY_THRESHOLD_CONFIG_PATH = PROJECT_DIR / "reports" / "analysis_outputs" / "training_runs_binary_thoracolumbar" / "binary_spine_threshold_config.json"

INFERENCE_IMAGE_DIR = PROJECT_DIR / "outputs" / "inference_inputs"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "final_inference_pipeline_thoracolumbar"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [INDEX_PATH, DICT_PATH, MANIFEST_PATH, BINARY_GROUP_MAP_PATH, BINARY_MODEL_PATH, MULTICLASS_MODEL_PATH, LAST_VISIBLE_MODEL_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"No existe archivo requerido: {path}")

if not INFERENCE_IMAGE_DIR.exists():
    INFERENCE_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Se creo INFERENCE_IMAGE_DIR: {INFERENCE_IMAGE_DIR}")

SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
PIN_MEMORY = False

IMG_SIZE_BINARY = (512, 256)
IMG_SIZE_MULTICLASS = (640, 320)
IMG_SIZE_LAST = (384, 192)
BINARY_THRESHOLD = 0.50
ROI_PAD_X = 28
ROI_PAD_Y = 44
MIN_FOREGROUND_PIXELS = 24
IGNORE_INDEX = 255

PROFILE_BINS = 24
PRESENCE_THRESHOLD_PIXELS = 40
ASSUMED_FIRST_VISIBLE_IDX = 0
SAVE_MASKS = True
SAVE_FIGURES = True
MAX_VIS_PREVIEW = 10
USE_MULTICLASS_TTA = True
LAST_EXPECTATION_BLEND = 0.30
LAST_HEURISTIC_BLEND = 0.20

MIN_COMPONENT_PIXELS = 48
SECONDARY_COMPONENT_RATIO = 0.35
MAX_LABEL_GAP_FOR_MAIN_BAND = 2
ALLOW_MARGIN_LABELS = 1
MIN_AREA_RATIO_TO_KEEP_EDGE_OUTLIER = 0.45
MONOTONIC_TOLERANCE_PX = 6.0
RESTORE_EDGE_MIN_AREA_RATIO = 0.30
RESTORE_NEIGHBOR_DISTANCE = 1

if BINARY_THRESHOLD_CONFIG_PATH.exists():
    binary_threshold_config = json.loads(BINARY_THRESHOLD_CONFIG_PATH.read_text(encoding="utf-8"))
    BINARY_THRESHOLD = float(binary_threshold_config.get("selected_threshold", BINARY_THRESHOLD))

print("CWD:", CWD)
print("ROOT:", ROOT)


CWD: /content/drive/MyDrive/DataRadriografias
ROOT: /content/drive/MyDrive/DataRadriografias/data/Scoliosis_Dataset


In [15]:
from __future__ import annotations

import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

#CWD = Path.cwd()

ROOT_CANDIDATES = [
    Path.cwd() / "data" / "Scoliosis_Dataset",
    Path.cwd().parent / "data" / "Scoliosis_Dataset",
    Path.cwd().parent.parent / "data" / "Scoliosis_Dataset",
    Path.cwd() / "Scoliosis_Dataset",
    Path.cwd().parent / "Scoliosis_Dataset",
]

REQUIRED_FILES = [
    'indice_dataset.csv',
    'diccionario_etiquetas_T1_T12_L1_L5.json',
]


def is_valid_root(path: Path) -> bool:
    return path.exists() and all((path / rel).exists() for rel in REQUIRED_FILES)


ROOT = next((p for p in ROOT_CANDIDATES if is_valid_root(p)), None)
assert ROOT is not None, (
    'No se pudo localizar la carpeta data/Scoliosis_Dataset con los archivos esperados. '
    f'Directorio actual: {CWD}'
)

ROOT = ROOT.resolve()
PROJECT_DIR_CANDIDATES = [ROOT.parent.parent, *ROOT.parents, CWD, *CWD.parents]
PROJECT_DIR = next(
    (
        path
        for path in PROJECT_DIR_CANDIDATES
        if (path / "notebooks").exists() and (path / "data").exists()
    ),
    ROOT.parent.parent,
)
REPORTS_DIR = PROJECT_DIR / "reports"
ANALYSIS_DIR = REPORTS_DIR / "analysis_outputs"
OUTPUTS_DIR = PROJECT_DIR / "outputs"
MODELS_DIR = PROJECT_DIR / "models"
for directory in [REPORTS_DIR, ANALYSIS_DIR, OUTPUTS_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SEARCH_BASES = [CWD.resolve(), ROOT, *ROOT.parents]


def resolve_dataset_path(path_value: str | Path) -> Path:
    raw = Path(str(path_value))
    candidates: list[Path] = []

    if raw.is_absolute():
        candidates.append(raw)
    else:
        candidates.append(raw)
        candidates.extend(base / raw for base in SEARCH_BASES)

    parts = raw.parts
    if 'Scoliosis_Dataset' in parts:
            idx = parts.index('Scoliosis_Dataset')
            trimmed = Path(*parts[idx + 1:])
            if str(trimmed) not in {'', '.'}:
                candidates.append(ROOT / trimmed)

    seen: set[str] = set()
    unique_candidates: list[Path] = []
    for candidate in candidates:
        key = str(candidate)
        if key not in seen:
            seen.add(key)
            unique_candidates.append(candidate)

    for candidate in unique_candidates:
        if candidate.exists():
            return candidate.resolve()

    return unique_candidates[-1].resolve()

INDEX_PATH = ROOT / 'indice_dataset.csv'
DICT_PATH = ROOT / 'diccionario_etiquetas_T1_T12_L1_L5.json'
MANIFEST_PATH = PROJECT_DIR / 'reports' / 'analysis_outputs' / 'training_manifest_thoracolumbar.csv'
BINARY_GROUP_MAP_PATH = PROJECT_DIR / 'reports' / 'analysis_outputs' / 'training_runs_binary_thoracolumbar' / 'binary_spine_group_partition_map.csv'
BINARY_MODEL_PATH = PROJECT_DIR / 'models' / 'binary_spine_thoracolumbar_best.pt'
MULTICLASS_MODEL_PATH = PROJECT_DIR / 'models' / 'thoracolumbar_partial_cascade_explained_best.pt'
LAST_VISIBLE_MODEL_PATH = PROJECT_DIR / 'models' / 'last_visible_estimator_thoracolumbar_best.pt'

INFERENCE_IMAGE_DIR = PROJECT_DIR / 'outputs' / 'inference_inputs'
OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'final_inference_pipeline_thoracolumbar'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [INDEX_PATH, DICT_PATH, MANIFEST_PATH, BINARY_GROUP_MAP_PATH, BINARY_MODEL_PATH, MULTICLASS_MODEL_PATH, LAST_VISIBLE_MODEL_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'No existe archivo requerido: {path}')

if not INFERENCE_IMAGE_DIR.exists():
    raise FileNotFoundError(
        f'No existe INFERENCE_IMAGE_DIR={INFERENCE_IMAGE_DIR}. Crea la carpeta y sube alli tus radiografias.'
    )

SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
PIN_MEMORY = False

IMG_SIZE_BINARY = (512, 256)
IMG_SIZE_MULTICLASS = (640, 320)
IMG_SIZE_LAST = (384, 192)
BINARY_THRESHOLD = 0.50
ROI_PAD_X = 28
ROI_PAD_Y = 44
MIN_FOREGROUND_PIXELS = 24
IGNORE_INDEX = 255

PROFILE_BINS = 24
PRESENCE_THRESHOLD_PIXELS = 40
ASSUMED_FIRST_VISIBLE_IDX = 0  # T1
SAVE_MASKS = True
SAVE_FIGURES = True
MAX_VIS_PREVIEW = 10

print('CWD:', CWD)
print('ROOT:', ROOT)
print('DEVICE:', DEVICE)
print('INFERENCE_IMAGE_DIR:', INFERENCE_IMAGE_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

CWD: /content/drive/MyDrive/DataRadriografias
ROOT: /content/drive/MyDrive/DataRadriografias/data/Scoliosis_Dataset
DEVICE: cuda
INFERENCE_IMAGE_DIR: /content/drive/MyDrive/DataRadriografias/outputs/inference_inputs
OUTPUT_DIR: /content/drive/MyDrive/DataRadriografias/outputs/final_inference_pipeline_thoracolumbar


## 2. Metadata y clases del problema

Esta seccion solo se usa para:

- reconstruir las clases del problema
- recalcular la normalizacion de features auxiliares del modelo final

In [16]:
index_df_raw = pd.read_csv(INDEX_PATH)
manifest_df = pd.read_csv(MANIFEST_PATH)
group_map_df = pd.read_csv(BINARY_GROUP_MAP_PATH)
with open(DICT_PATH, 'r', encoding='utf-8') as f:
    labels_dict = json.load(f)

index_col_map = {
    'grupo': 'split',
    'imagen': 'image',
    'id_paciente': 'patient_id',
    'ruta_radiografia': 'radiograph_path',
    'ruta_mascara_binaria': 'label_binary_path',
    'ruta_mascara_multiclase_id_png': 'multiclass_id_png',
}
index_df = index_df_raw.rename(columns=index_col_map).copy()

final_multiclass_map = {int(k): v for k, v in labels_dict['mascara_multiclase_id_png'].items()}
class_names = [final_multiclass_map[i] for i in range(len(final_multiclass_map))]
num_classes = len(class_names)
canonical_labels = [f'T{i}' for i in range(1, 13)] + [f'L{i}' for i in range(1, 6)]
label_to_class_id = {label: idx for idx, label in enumerate(class_names)}

join_cols = ['split', 'image', 'patient_id', 'radiograph_path']
dataset_subset = index_df[join_cols + ['label_binary_path', 'multiclass_id_png']].copy()
table = manifest_df.merge(dataset_subset, on=join_cols, how='left', suffixes=('', '_idx'))
table['multiclass_mask_path'] = table['mask_path'].fillna(table['multiclass_id_png'])
table['radiograph_path_abs'] = table['radiograph_path'].apply(lambda rel: str(resolve_dataset_path(rel)))
table['binary_mask_path_abs'] = table['label_binary_path'].apply(lambda rel: str(resolve_dataset_path(rel)))

for col in ['usable_for_thoracolumbar_core', 'usable_for_thoracolumbar_partial', 'needs_annotation_review']:
    if col in table.columns:
        table[col] = table[col].map(
            lambda x: x if isinstance(x, bool) else str(x).strip().lower() == 'true'
        )

group_partition_map = group_map_df.drop_duplicates().set_index('group_id_for_split')['partition'].to_dict()
table['partition'] = table['group_id_for_split'].map(group_partition_map)

train_norm_df = table.loc[
    table['usable_for_thoracolumbar_partial'] & ~table['needs_annotation_review'] & table['partition'].eq('train')
].copy().reset_index(drop=True)

print('Train samples for aux normalization:', len(train_norm_df))

Train samples for aux normalization: 143


## 3. Utilidades de imagen y arquitecturas del pipeline

In [17]:
def read_gray(path: str) -> np.ndarray:
    return np.array(Image.open(path).convert('L'))


def resize_image(arr: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    return np.array(Image.fromarray(arr).resize((size[1], size[0]), resample=Image.BILINEAR))


def resize_mask(arr: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    return np.array(Image.fromarray(arr.astype(np.uint8)).resize((size[1], size[0]), resample=Image.NEAREST))


def build_binary_mask(path: str, size: tuple[int, int] | None = None) -> np.ndarray:
    mask = read_gray(path)
    mask = (mask >= 127).astype(np.uint8)
    if size is not None:
        mask = resize_mask(mask, size)
    return mask


def bbox_from_mask(mask: np.ndarray, min_foreground_pixels: int = 24) -> tuple[int, int, int, int] | None:
    ys, xs = np.where(mask > 0)
    if len(xs) < min_foreground_pixels:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1


def clamp_bbox(bbox: tuple[int, int, int, int], image_shape: tuple[int, int]) -> tuple[int, int, int, int]:
    h, w = image_shape
    x0, y0, x1, y1 = bbox
    x0 = max(0, min(x0, w - 1))
    y0 = max(0, min(y0, h - 1))
    x1 = max(x0 + 1, min(x1, w))
    y1 = max(y0 + 1, min(y1, h))
    return x0, y0, x1, y1


def expand_bbox(bbox: tuple[int, int, int, int], image_shape: tuple[int, int], pad_x: int = 28, pad_y: int = 44) -> tuple[int, int, int, int]:
    x0, y0, x1, y1 = bbox
    return clamp_bbox((x0 - pad_x, y0 - pad_y, x1 + pad_x, y1 + pad_y), image_shape)


def scale_bbox(bbox: tuple[int, int, int, int], src_shape: tuple[int, int], dst_shape: tuple[int, int]) -> tuple[int, int, int, int]:
    src_h, src_w = src_shape
    dst_h, dst_w = dst_shape
    x0, y0, x1, y1 = bbox
    sx = dst_w / src_w
    sy = dst_h / src_h
    scaled = (int(round(x0 * sx)), int(round(y0 * sy)), int(round(x1 * sx)), int(round(y1 * sy)))
    return clamp_bbox(scaled, dst_shape)


def crop_array(arr: np.ndarray, bbox: tuple[int, int, int, int]) -> np.ndarray:
    x0, y0, x1, y1 = bbox
    return arr[y0:y1, x0:x1]


def normalize_image(image_2d: np.ndarray) -> np.ndarray:
    mean = float(image_2d.mean())
    std = float(image_2d.std())
    if std < 1e-6:
        return image_2d - mean
    return (image_2d - mean) / std


def build_coordinate_channels(height: int, width: int) -> np.ndarray:
    y_coords = np.linspace(0.0, 1.0, height, dtype=np.float32)[:, None]
    x_coords = np.linspace(0.0, 1.0, width, dtype=np.float32)[None, :]
    y_map = np.repeat(y_coords, width, axis=1)
    x_map = np.repeat(x_coords, height, axis=0)
    return np.stack([y_map, x_map], axis=0)


class DoubleConvBinary(nn.Module):
    def __init__(self, c_in: int, c_out: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class BinaryUNetSmall(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, base: int = 32):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.e1 = DoubleConvBinary(in_channels, base)
        self.e2 = DoubleConvBinary(base, base * 2)
        self.e3 = DoubleConvBinary(base * 2, base * 4)
        self.e4 = DoubleConvBinary(base * 4, base * 8)
        self.b = DoubleConvBinary(base * 8, base * 16)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, kernel_size=2, stride=2)
        self.d4 = DoubleConvBinary(base * 16, base * 8)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, kernel_size=2, stride=2)
        self.d3 = DoubleConvBinary(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.d2 = DoubleConvBinary(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.d1 = DoubleConvBinary(base * 2, base)
        self.head = nn.Conv2d(base, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.b(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), e4], dim=1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        return self.head(d1)


class DoubleConv(nn.Module):
    def __init__(self, c_in: int, c_out: int, dropout: float = 0.0):
        super().__init__()
        layers = [
            nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class UNetEnhanced(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, base: int = 48, dropout: float = 0.10):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.e1 = DoubleConv(in_channels, base, dropout=0.0)
        self.e2 = DoubleConv(base, base * 2, dropout=0.0)
        self.e3 = DoubleConv(base * 2, base * 4, dropout=0.0)
        self.e4 = DoubleConv(base * 4, base * 8, dropout=dropout * 0.5)
        self.b = DoubleConv(base * 8, base * 16, dropout=dropout)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, kernel_size=2, stride=2)
        self.d4 = DoubleConv(base * 16, base * 8, dropout=dropout * 0.5)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, kernel_size=2, stride=2)
        self.d3 = DoubleConv(base * 8, base * 4, dropout=0.0)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.d2 = DoubleConv(base * 4, base * 2, dropout=0.0)
        self.u1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.d1 = DoubleConv(base * 2, base, dropout=0.0)
        self.head = nn.Conv2d(base, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.b(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), e4], dim=1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        return self.head(d1)


class ConvBlock(nn.Module):
    def __init__(self, c_in: int, c_out: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class LastVisibleEstimator(nn.Module):
    def __init__(self, aux_dim: int, num_labels: int = 17, dropout: float = 0.25):
        super().__init__()
        self.image_encoder = nn.Sequential(
            ConvBlock(3, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
        )
        self.image_pool = nn.AdaptiveAvgPool2d(1)
        self.image_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 192),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.aux_head = nn.Sequential(
            nn.Linear(aux_dim, 192),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(192, 96),
            nn.ReLU(inplace=True),
        )
        self.fusion = nn.Sequential(
            nn.Linear(192 + 96, 160),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(160, num_labels),
        )

    def forward(self, image: torch.Tensor, aux: torch.Tensor) -> torch.Tensor:
        image_feat = self.image_encoder(image)
        image_feat = self.image_pool(image_feat)
        image_feat = self.image_head(image_feat)
        aux_feat = self.aux_head(aux)
        fused = torch.cat([image_feat, aux_feat], dim=1)
        return self.fusion(fused)

## 4. Carga de modelos

In [18]:
binary_model = BinaryUNetSmall(in_channels=1, out_channels=1).to(DEVICE)
binary_model.load_state_dict(torch.load(BINARY_MODEL_PATH, map_location=DEVICE))
binary_model.eval()

multiclass_model = UNetEnhanced(in_channels=3, out_channels=num_classes, base=48, dropout=0.10).to(DEVICE)
multiclass_model.load_state_dict(torch.load(MULTICLASS_MODEL_PATH, map_location=DEVICE))
multiclass_model.eval()

placeholder_aux_dim = 146
last_model = LastVisibleEstimator(aux_dim=placeholder_aux_dim, num_labels=len(canonical_labels), dropout=0.25).to(DEVICE)
last_model.load_state_dict(torch.load(LAST_VISIBLE_MODEL_PATH, map_location=DEVICE))
last_model.eval()

print("Models loaded.")


Models loaded.


## 5. Features auxiliares y normalizacion del modelo final

Para el estimador final necesitamos reproducir la normalizacion de features
auxiliares usando las muestras de entrenamiento.

In [19]:
def predict_binary_bbox_from_image(image_raw: np.ndarray) -> tuple[int, int, int, int] | None:
    image_resized = resize_image(image_raw, IMG_SIZE_BINARY).astype(np.float32) / 255.0
    image_tensor = torch.tensor(image_resized[None, None, ...], dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        logits = binary_model(image_tensor)
        pred_mask_small = (torch.sigmoid(logits)[0, 0].detach().cpu().numpy() >= BINARY_THRESHOLD).astype(np.uint8)
    return bbox_from_mask(pred_mask_small, min_foreground_pixels=MIN_FOREGROUND_PIXELS)


def infer_multiclass_on_bbox(image_raw: np.ndarray, bbox: tuple[int, int, int, int]) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    image_crop = crop_array(image_raw, bbox)
    image_crop = resize_image(image_crop, IMG_SIZE_MULTICLASS).astype(np.float32) / 255.0
    image_crop = normalize_image(image_crop)
    coords = build_coordinate_channels(IMG_SIZE_MULTICLASS[0], IMG_SIZE_MULTICLASS[1])
    image_channels = np.concatenate([np.expand_dims(image_crop, axis=0), coords], axis=0)
    image_tensor = torch.tensor(image_channels[None, ...], dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        logits = multiclass_model(image_tensor)
        probs = torch.softmax(logits, dim=1)
        if USE_MULTICLASS_TTA:
            flipped_input = torch.flip(image_tensor, dims=[3])
            flipped_logits = multiclass_model(flipped_input)
            flipped_probs = torch.softmax(flipped_logits, dim=1)
            flipped_probs = torch.flip(flipped_probs, dims=[3])
            probs = 0.5 * (probs + flipped_probs)
        probs_np = probs[0].detach().cpu().numpy().astype(np.float32)
        pred_mask = np.argmax(probs_np, axis=0).astype(np.int64)
    return image_crop, pred_mask, probs_np


def extract_aux_features_from_prediction(pred_mask: np.ndarray, prob_map: np.ndarray) -> np.ndarray:
    h, w = pred_mask.shape
    fg_mask = (pred_mask > 0).astype(np.float32)
    total_fg = float(fg_mask.sum()) + 1e-6

    presence = []
    area_ratio = []
    centroid_y = []
    y_min_norm = []
    y_max_norm = []
    height_span_norm = []
    mean_confidence = []
    for label in canonical_labels:
        class_id = label_to_class_id[label]
        class_mask = pred_mask == class_id
        area = float(class_mask.sum())
        presence.append(1.0 if area >= PRESENCE_THRESHOLD_PIXELS else 0.0)
        area_ratio.append(area / total_fg)
        if area > 0:
            ys, _ = np.where(class_mask)
            centroid_y.append(float(np.mean(ys) / max(h - 1, 1)))
            y_min_norm.append(float(np.min(ys) / max(h - 1, 1)))
            y_max_norm.append(float(np.max(ys) / max(h - 1, 1)))
            height_span_norm.append(float((np.max(ys) - np.min(ys) + 1) / max(h, 1)))
            mean_confidence.append(float(prob_map[class_id][class_mask].mean()))
        else:
            centroid_y.append(0.0)
            y_min_norm.append(0.0)
            y_max_norm.append(0.0)
            height_span_norm.append(0.0)
            mean_confidence.append(0.0)

    pred_present_indices = [i for i, p in enumerate(presence) if p > 0.5]
    min_present_idx = float(min(pred_present_indices)) if pred_present_indices else 0.0
    max_present_idx = float(max(pred_present_indices)) if pred_present_indices else 0.0
    num_present = float(len(pred_present_indices))

    row_profile = fg_mask.sum(axis=1).astype(np.float32)
    if row_profile.max() > 0:
        row_profile = row_profile / row_profile.max()
    binned_profile = np.array_split(row_profile, PROFILE_BINS)
    profile_features = [float(chunk.mean()) for chunk in binned_profile]

    return np.array(
        presence
        + area_ratio
        + centroid_y
        + y_min_norm
        + y_max_norm
        + height_span_norm
        + mean_confidence
        + [
            min_present_idx / (len(canonical_labels) - 1),
            max_present_idx / (len(canonical_labels) - 1),
            num_present / len(canonical_labels),
        ]
        + profile_features,
        dtype=np.float32,
    )


def estimate_last_visible_from_mask(pred_mask: np.ndarray) -> int:
    present_indices = [
        canonical_labels.index(class_names[class_id])
        for class_id in sorted(int(x) for x in np.unique(pred_mask) if int(x) > 0)
        if class_names[class_id] in canonical_labels
    ]
    if not present_indices:
        return 0
    return int(max(present_indices))


train_aux_features = []
for _, row in train_norm_df.iterrows():
    image_raw = read_gray(row["radiograph_path_abs"])
    gt_binary = build_binary_mask(row["binary_mask_path_abs"], size=None)
    bbox = bbox_from_mask(gt_binary, min_foreground_pixels=MIN_FOREGROUND_PIXELS)
    if bbox is None:
        bbox = (0, 0, image_raw.shape[1], image_raw.shape[0])
    else:
        bbox = expand_bbox(bbox, image_shape=image_raw.shape, pad_x=ROI_PAD_X, pad_y=ROI_PAD_Y)
    _, pred_mask, prob_map = infer_multiclass_on_bbox(image_raw, bbox)
    train_aux_features.append(extract_aux_features_from_prediction(pred_mask, prob_map))

train_aux_features = np.stack(train_aux_features, axis=0)
aux_mean = train_aux_features.mean(axis=0).astype(np.float32)
aux_std = train_aux_features.std(axis=0).astype(np.float32)
aux_std = np.where(aux_std < 1e-6, 1.0, aux_std)

print("train_aux_features:", train_aux_features.shape)


train_aux_features: (143, 146)


## 6. Inferencia del pipeline completo sobre imagenes nuevas

In [22]:
def clip_mask_to_last_idx(mask_2d: np.ndarray, first_idx: int, last_idx: int) -> np.ndarray:
    first_idx = int(first_idx)
    last_idx = int(max(last_idx, first_idx))
    allowed_labels = canonical_labels[first_idx:last_idx + 1]
    allowed_ids = {label_to_class_id[label] for label in allowed_labels}
    out = np.zeros_like(mask_2d, dtype=np.int64)
    for class_id in allowed_ids:
        out[mask_2d == class_id] = class_id
    return out


def keep_reliable_components_per_class(mask_2d: np.ndarray) -> np.ndarray:
    cleaned = np.zeros_like(mask_2d, dtype=np.int64)
    for class_id in range(1, num_classes):
        class_mask = (mask_2d == class_id).astype(np.uint8)
        labeled, num_components = ndi.label(class_mask)
        if num_components == 0:
            continue
        component_ids = np.arange(1, num_components + 1)
        areas = np.array(ndi.sum(class_mask, labeled, index=component_ids), dtype=np.float64)
        largest_area = float(areas.max())
        keep_ids = []
        for comp_id, area in zip(component_ids, areas):
            if area < MIN_COMPONENT_PIXELS:
                continue
            if area >= largest_area * SECONDARY_COMPONENT_RATIO:
                keep_ids.append(int(comp_id))
        if not keep_ids and largest_area >= MIN_COMPONENT_PIXELS:
            keep_ids = [int(component_ids[np.argmax(areas)])]
        for comp_id in keep_ids:
            cleaned[labeled == comp_id] = class_id
    return cleaned


def class_stats_from_mask(mask_2d: np.ndarray) -> pd.DataFrame:
    rows = []
    total_fg = float((mask_2d > 0).sum()) + 1e-6
    for class_id in range(1, num_classes):
        class_mask = mask_2d == class_id
        if not class_mask.any():
            continue
        ys, xs = np.where(class_mask)
        rows.append({
            "class_id": class_id,
            "class_name": class_names[class_id],
            "label_index": canonical_labels.index(class_names[class_id]),
            "area_pixels": int(class_mask.sum()),
            "area_ratio": float(class_mask.sum() / total_fg),
            "centroid_y": float(np.mean(ys)),
            "y_min": int(ys.min()),
            "y_max": int(ys.max()),
        })
    if not rows:
        return pd.DataFrame(columns=["class_id", "class_name", "label_index", "area_pixels", "area_ratio", "centroid_y", "y_min", "y_max"])
    return pd.DataFrame(rows).sort_values("label_index").reset_index(drop=True)


def build_label_blocks(label_indices: list[int], gap_tolerance: int = 2) -> list[list[int]]:
    if not label_indices:
        return []
    blocks = [[label_indices[0]]]
    for idx in label_indices[1:]:
        if idx - blocks[-1][-1] <= gap_tolerance + 1:
            blocks[-1].append(idx)
        else:
            blocks.append([idx])
    return blocks


def select_dominant_label_band(stats_df: pd.DataFrame) -> tuple[int | None, int | None]:
    if stats_df.empty:
        return None, None
    indices = stats_df["label_index"].tolist()
    blocks = build_label_blocks(indices, gap_tolerance=MAX_LABEL_GAP_FOR_MAIN_BAND)
    scored = []
    for block in blocks:
        block_df = stats_df.loc[stats_df["label_index"].isin(block)]
        score = (len(block), float(block_df["area_pixels"].sum()), -float(block_df["label_index"].min()))
        scored.append((score, block[0], block[-1]))
    scored = sorted(scored, reverse=True)
    return int(scored[0][1]), int(scored[0][2])


def trim_edge_outliers(mask_2d: np.ndarray, stats_df: pd.DataFrame, start_idx: int | None, end_idx: int | None) -> np.ndarray:
    if stats_df.empty or start_idx is None or end_idx is None:
        return mask_2d
    out = mask_2d.copy()
    median_area = float(stats_df["area_pixels"].median()) if not stats_df.empty else 0.0
    allowed_start = max(0, start_idx - ALLOW_MARGIN_LABELS)
    allowed_end = min(len(canonical_labels) - 1, end_idx + ALLOW_MARGIN_LABELS)
    for _, row in stats_df.iterrows():
        label_index = int(row["label_index"])
        if allowed_start <= label_index <= allowed_end:
            continue
        if median_area <= 0:
            continue
        if float(row["area_pixels"]) < median_area * MIN_AREA_RATIO_TO_KEEP_EDGE_OUTLIER:
            out[out == int(row["class_id"])] = 0
    return out


def enforce_monotonic_centroids(mask_2d: np.ndarray) -> np.ndarray:
    out = mask_2d.copy()
    while True:
        stats_df = class_stats_from_mask(out)
        if len(stats_df) <= 1:
            break
        violation_found = False
        rows = stats_df.to_dict(orient="records")
        for prev_row, curr_row in zip(rows[:-1], rows[1:]):
            if float(curr_row["centroid_y"]) + MONOTONIC_TOLERANCE_PX < float(prev_row["centroid_y"]):
                drop_class = int(prev_row["class_id"]) if float(prev_row["area_pixels"]) < float(curr_row["area_pixels"]) else int(curr_row["class_id"])
                out[out == drop_class] = 0
                violation_found = True
                break
        if not violation_found:
            break
    return out


def restore_supported_edge_labels(mask_2d: np.ndarray, reference_mask: np.ndarray, start_idx: int | None, end_idx: int | None) -> np.ndarray:
    if start_idx is None or end_idx is None:
        return mask_2d
    out = mask_2d.copy()
    ref_stats = class_stats_from_mask(reference_mask)
    final_stats = class_stats_from_mask(out)
    if ref_stats.empty:
        return out
    median_area = float(ref_stats["area_pixels"].median()) if not ref_stats.empty else 0.0
    kept_indices = set(final_stats["label_index"].tolist())
    boundary_candidates = {max(0, start_idx - 1), min(len(canonical_labels) - 1, end_idx + 1)}
    for _, row in ref_stats.iterrows():
        label_index = int(row["label_index"])
        if label_index in kept_indices:
            continue
        if label_index not in boundary_candidates:
            continue
        if abs(label_index - start_idx) > RESTORE_NEIGHBOR_DISTANCE and abs(label_index - end_idx) > RESTORE_NEIGHBOR_DISTANCE:
            continue
        if median_area <= 0:
            continue
        if float(row["area_pixels"]) < median_area * RESTORE_EDGE_MIN_AREA_RATIO:
            continue
        class_id = int(row["class_id"])
        out[reference_mask == class_id] = class_id
    return out


def postprocess_mask(mask_2d: np.ndarray) -> np.ndarray:
    cleaned = keep_reliable_components_per_class(mask_2d)
    stats_initial = class_stats_from_mask(cleaned)
    start_idx, end_idx = select_dominant_label_band(stats_initial)
    trimmed = trim_edge_outliers(cleaned, stats_initial, start_idx, end_idx)
    monotonic = enforce_monotonic_centroids(trimmed)
    restored = restore_supported_edge_labels(monotonic, cleaned, start_idx, end_idx)
    return restored


def blend_last_prediction(logits: torch.Tensor, heuristic_last_idx: int) -> int:
    probs = torch.softmax(logits, dim=1)[0]
    argmax_idx = int(torch.argmax(probs).item())
    class_axis = torch.arange(probs.shape[0], dtype=torch.float32, device=probs.device)
    expected_idx = float((probs * class_axis).sum().item())
    blended = (
        (1.0 - LAST_EXPECTATION_BLEND - LAST_HEURISTIC_BLEND) * argmax_idx
        + LAST_EXPECTATION_BLEND * expected_idx
        + LAST_HEURISTIC_BLEND * float(heuristic_last_idx)
    )
    return int(np.clip(round(blended), ASSUMED_FIRST_VISIBLE_IDX, len(canonical_labels) - 1))


def present_labels_from_mask(mask_2d: np.ndarray) -> list[str]:
    ids = sorted(int(x) for x in np.unique(mask_2d) if int(x) > 0)
    return [class_names[i] for i in ids]


def collect_input_images(input_dir: Path) -> list[Path]:
    return sorted(
        path for path in input_dir.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    )


image_paths = collect_input_images(INFERENCE_IMAGE_DIR)
if not image_paths:
    raise FileNotFoundError(f"No se encontraron imagenes soportadas en {INFERENCE_IMAGE_DIR}")

print("Images found:", len(image_paths))


Images found: 1


In [23]:
def save_mask_png(mask_2d: np.ndarray, path: Path) -> None:
    Image.fromarray(mask_2d.astype(np.uint8)).save(path)


records = []
preview_rows = []

for image_path in image_paths:
    image_raw = read_gray(str(image_path))
    image_shape = image_raw.shape

    bbox_small = predict_binary_bbox_from_image(image_raw)
    if bbox_small is None:
        bbox = (0, 0, image_shape[1], image_shape[0])
        roi_source = "pred_binary_fallback_full_image"
    else:
        bbox = scale_bbox(bbox_small, src_shape=IMG_SIZE_BINARY, dst_shape=image_shape)
        bbox = expand_bbox(bbox, image_shape=image_shape, pad_x=ROI_PAD_X, pad_y=ROI_PAD_Y)
        roi_source = "pred_binary"

    image_crop, raw_mask, prob_map = infer_multiclass_on_bbox(image_raw, bbox)
    post_mask = postprocess_mask(raw_mask)
    aux_features = extract_aux_features_from_prediction(post_mask, prob_map)
    aux_norm = ((aux_features - aux_mean) / aux_std).astype(np.float32)
    coords_last = build_coordinate_channels(IMG_SIZE_LAST[0], IMG_SIZE_LAST[1])
    image_small = resize_image((image_crop * 255.0).astype(np.uint8), IMG_SIZE_LAST).astype(np.float32) / 255.0
    image_small = normalize_image(image_small)
    image_last = np.concatenate([np.expand_dims(image_small, axis=0), coords_last], axis=0)
    heuristic_last_idx = estimate_last_visible_from_mask(post_mask)

    image_tensor = torch.tensor(image_last[None, ...], dtype=torch.float32, device=DEVICE)
    aux_tensor = torch.tensor(aux_norm[None, ...], dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        logits = last_model(image_tensor, aux_tensor)
    pred_last_idx = blend_last_prediction(logits, heuristic_last_idx)
    final_mask = clip_mask_to_last_idx(post_mask, ASSUMED_FIRST_VISIBLE_IDX, pred_last_idx)

    labels_raw = present_labels_from_mask(raw_mask)
    labels_post = present_labels_from_mask(post_mask)
    labels_final = present_labels_from_mask(final_mask)
    final_last_label = canonical_labels[pred_last_idx]

    mask_path = OUTPUT_DIR / f"{image_path.stem}_final_mask.png"
    preview_path = OUTPUT_DIR / f"{image_path.stem}_preview.png"

    if SAVE_MASKS:
        save_mask_png(final_mask, mask_path)

    fig = None
    if SAVE_FIGURES:
        fig, axes = plt.subplots(1, 4, figsize=(18, 5))
        axes[0].imshow(image_raw, cmap="gray")
        axes[0].set_title("Radiografia")
        axes[1].imshow(raw_mask, cmap="nipy_spectral", vmin=0, vmax=num_classes - 1)
        axes[1].set_title("Raw multiclass")
        axes[2].imshow(post_mask, cmap="nipy_spectral", vmin=0, vmax=num_classes - 1)
        axes[2].set_title("Postprocess")
        axes[3].imshow(final_mask, cmap="nipy_spectral", vmin=0, vmax=num_classes - 1)
        axes[3].set_title(f"Final clip <= {final_last_label}")
        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        fig.savefig(preview_path, dpi=160, bbox_inches="tight")
        plt.close(fig)

    records.append({
        "image_name": image_path.name,
        "roi_source": roi_source,
        "bbox_x0": int(bbox[0]),
        "bbox_y0": int(bbox[1]),
        "bbox_x1": int(bbox[2]),
        "bbox_y1": int(bbox[3]),
        "heuristic_last_idx": int(heuristic_last_idx),
        "pred_last_idx": int(pred_last_idx),
        "pred_last_label": final_last_label,
        "raw_labels": ", ".join(labels_raw),
        "post_labels": ", ".join(labels_post),
        "final_labels": ", ".join(labels_final),
        "raw_label_count": len(labels_raw),
        "post_label_count": len(labels_post),
        "final_label_count": len(labels_final),
        "mask_output_path": str(mask_path) if SAVE_MASKS else "",
        "preview_output_path": str(preview_path) if SAVE_FIGURES else "",
    })

    preview_rows.append({
        "image_name": image_path.name,
        "pred_last_label": final_last_label,
        "final_labels": ", ".join(labels_final),
    })

results_df = pd.DataFrame(records)
results_path = OUTPUT_DIR / "inference_results.csv"
results_df.to_csv(results_path, index=False)

print("Resultados guardados en:", results_path)
display(results_df.head(MAX_VIS_PREVIEW))


Resultados guardados en: /content/drive/MyDrive/DataRadriografias/outputs/final_inference_pipeline_thoracolumbar/inference_results.csv


,image_name,roi_source,bbox_x0,bbox_y0,bbox_x1,bbox_y1,heuristic_last_idx,pred_last_idx,pred_last_label,raw_labels,post_labels,final_labels,raw_label_count,post_label_count,final_label_count,mask_output_path,preview_output_path
0,N_59.jpg,pred_binary,13,0,145,806,16,16,L5,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...","T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...","T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...",17,17,17,/content/drive/MyDrive/DataRadriografias/outpu...,/content/drive/MyDrive/DataRadriografias/outpu...


## 7. Vista rapida de resultados

In [24]:
display(
    results_df[
        ["image_name", "pred_last_label", "raw_label_count", "final_label_count", "final_labels"]
    ].sort_values(["pred_last_label", "final_label_count"])
)

,image_name,pred_last_label,raw_label_count,final_label_count,final_labels
0,N_59.jpg,L5,17,17,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ..."


## 8. Conclusiones de uso

Este notebook representa la mejor forma actual de usar el pipeline:

- ROI binaria para ubicar la columna
- segmentacion multiclase para vertebras
- estimacion especializada del extremo visible
- clipping anatomico para reducir sobreprediccion

Si luego quieres convertir esto en una aplicacion o script productivo, esta es la
base recomendada.